# ❤️ Heart Disease Detection — Model Training & Comparison
> **MA411 — Expert Systems Project**  
> Decision Tree Classifier + Rule-Based Expert System + Performance Comparison

---

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'utils'))
sys.path.insert(0, os.path.join('..', 'rule_based_system'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix, roc_curve, auc)
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
print('✔ All imports successful')

## 2. Load & Inspect Cleaned Data

In [ ]:
df = pd.read_csv('../data/cleaned_data.csv').fillna(lambda x: x.median())
print(f'Cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f"Target distribution:\n{df['target'].value_counts().to_string()}")
df.head()

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set     : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'Features     : {X_train.shape[1]}')

## 4. Decision Tree — Baseline Model

In [ ]:
# Baseline with default params
dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train, y_train)
y_pred_base = dt_baseline.predict(X_test)

print('=== Baseline Decision Tree ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_base):.2%}')
print(f'F1-Score  : {f1_score(y_test, y_pred_base):.2%}')
print(f'Tree depth: {dt_baseline.get_depth()}')
print(f'Leaves    : {dt_baseline.get_n_leaves()}')

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'max_depth'        : [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'criterion'        : ['gini', 'entropy']
}

gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
)
gs.fit(X_train, y_train)

print(f'Best parameters: {gs.best_params_}')
print(f'Best CV F1-Score: {gs.best_score_:.4f}')

clf = gs.best_estimator_

## 6. Evaluate Tuned Model

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
    'F1-Score' : f1_score(y_test, y_pred),
}

print('=== Tuned Decision Tree ===')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.2%}')
print()
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

## 7. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'],
            linewidths=1, linecolor='white', cbar=False)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#c0392b', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0,1], [0,1], 'k--', lw=1.2, alpha=0.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#c0392b')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Decision Tree', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()
print(f'AUC Score: {roc_auc:.4f}')

## 8. Cross-Validation Stability

In [ ]:
cv_scores = cross_val_score(clf, X, y, cv=10, scoring='f1')

plt.figure(figsize=(9, 3.5))
plt.bar(range(1, 11), cv_scores, color='#2980b9', alpha=0.8, edgecolor='white')
plt.axhline(cv_scores.mean(), color='#c0392b', linestyle='--', lw=2,
            label=f'Mean F1 = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
plt.xlabel('Fold')
plt.ylabel('F1-Score')
plt.title('10-Fold Cross-Validation F1 Scores', fontsize=12, fontweight='bold')
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print(f'Mean F1  : {cv_scores.mean():.4f}')
print(f'Std Dev  : {cv_scores.std():.4f}')
print(f'Min / Max: {cv_scores.min():.4f} / {cv_scores.max():.4f}')

## 9. Feature Importance

In [ ]:
importances = clf.feature_importances_
feat_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True)

colors = ['#c0392b' if v > 0.1 else '#3498db' if v > 0.05 else '#95a5a6'
          for v in feat_df['importance']]

plt.figure(figsize=(9, 6))
plt.barh(feat_df['feature'], feat_df['importance'], color=colors, alpha=0.85, edgecolor='white')
plt.xlabel('Importance Score')
plt.title('Feature Importances — Tuned Decision Tree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(feat_df.sort_values('importance', ascending=False).head(5).to_string(index=False))

## 10. Visualize the Decision Tree

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(clf,
          feature_names=X.columns.tolist(),
          class_names=['No Disease', 'Disease'],
          filled=True, rounded=True,
          max_depth=3,
          fontsize=9,
          impurity=False)
plt.title('Decision Tree (first 3 levels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Text representation
print(export_text(clf, feature_names=X.columns.tolist(), max_depth=3))

## 11. Expert System Evaluation on Test Set

In [ ]:
from rules import HeartDiseaseExpertSystem, PatientData

raw = pd.read_csv('../data/raw_data.csv').fillna(lambda c: c.median())
raw_test = raw.iloc[X_test.index] if hasattr(X_test, 'index') else raw.tail(len(X_test))

# Map normalized back for Expert System using raw values
raw_test = raw.sample(len(X_test), random_state=99).reset_index(drop=True)
y_test_es = raw_test['target'].values

es_preds = []
for _, row in raw_test.iterrows():
    engine = HeartDiseaseExpertSystem()
    engine.reset()
    engine.declare(PatientData(
        age=row['age'], sex=row['sex'], chol=row['chol'],
        trestbps=row['trestbps'], thalach=row['thalach'],
        fbs=row['fbs'], exang=row['exang'],
        cp=row['cp'], oldpeak=row['oldpeak'], ca=row['ca']
    ))
    engine.run()
    level = engine.get_risk_level()
    es_preds.append(1 if level in ['HIGH', 'MODERATE'] else 0)

es_acc  = accuracy_score(y_test_es, es_preds)
es_prec = precision_score(y_test_es, es_preds, zero_division=0)
es_rec  = recall_score(y_test_es, es_preds, zero_division=0)
es_f1   = f1_score(y_test_es, es_preds, zero_division=0)

print('=== Expert System on Test Set ===')
print(f'  Accuracy  : {es_acc:.2%}')
print(f'  Precision : {es_prec:.2%}')
print(f'  Recall    : {es_rec:.2%}')
print(f'  F1-Score  : {es_f1:.2%}')

## 12. Side-by-Side Comparison Chart

In [ ]:
comparison = pd.DataFrame({
    'Metric'        : ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Expert System' : [es_acc, es_prec, es_rec, es_f1],
    'Decision Tree' : [metrics['Accuracy'], metrics['Precision'],
                       metrics['Recall'], metrics['F1-Score']]
})

x = np.arange(len(comparison))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(x - w/2, comparison['Expert System'], w,
            label='Expert System', color='#3498db', alpha=0.85, edgecolor='white')
axes[0].bar(x + w/2, comparison['Decision Tree'], w,
            label='Decision Tree', color='#e74c3c', alpha=0.85, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison['Metric'])
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Score')
axes[0].set_title('Performance Comparison', fontsize=13, fontweight='bold')
axes[0].legend()
for i, (es, dt) in enumerate(zip(comparison['Expert System'], comparison['Decision Tree'])):
    axes[0].text(i - w/2, es + 0.02, f'{es:.2f}', ha='center', fontsize=9, color='#2c3e50')
    axes[0].text(i + w/2, dt + 0.02, f'{dt:.2f}', ha='center', fontsize=9, color='#2c3e50')

# Radar chart
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ax2 = fig.add_subplot(122, polar=True)
for label, values, color in [
    ('Expert System', comparison['Expert System'].tolist(), '#3498db'),
    ('Decision Tree', comparison['Decision Tree'].tolist(), '#e74c3c')
]:
    vals = values + values[:1]
    ax2.plot(angles, vals, 'o-', linewidth=2, color=color, label=label)
    ax2.fill(angles, vals, alpha=0.15, color=color)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories, fontsize=10)
ax2.set_ylim(0, 1)
ax2.set_title('Radar Chart', fontsize=13, fontweight='bold', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

print('\nFull comparison table:')
print(comparison.to_string(index=False))

## 13. Save Model

In [ ]:
joblib.dump(clf, '../ml_model/heart_disease_model.pkl')
pd.DataFrame([metrics]).to_csv('../reports/dt_metrics.csv', index=False)
print('✔ Model saved to: ../ml_model/heart_disease_model.pkl')
print('✔ Metrics saved to: ../reports/dt_metrics.csv')

## 14. Depth vs Accuracy Trade-off

In [ ]:
depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(f1_score(y_train, dt.predict(X_train)))
    test_scores.append(f1_score(y_test, dt.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(depths, train_scores, 'o-', color='#3498db', label='Train F1')
plt.plot(depths, test_scores,  's-', color='#c0392b', label='Test F1')
plt.axvline(clf.get_depth(), color='gray', linestyle='--', alpha=0.7,
            label=f'Best depth = {clf.get_depth()}')
plt.xlabel('Max Tree Depth')
plt.ylabel('F1-Score')
plt.title('Overfitting Analysis: Depth vs F1-Score', fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Final Summary

| System | Accuracy | Precision | Recall | F1 | Explainability | Data Needed |
|--------|:--------:|:---------:|:------:|:--:|:--------------:|:-----------:|
| **Expert System** | ~74% | ~71% | ~79% | ~75% | ✅ Full | ❌ None |
| **Decision Tree** | ~66–82% | ~62–82% | ~72–85% | ~67–83% | 🟡 Partial | ✅ Yes |

### Recommendation
A **hybrid deployment** is optimal:
- Expert System for first-pass triage (transparent, auditable by doctors)
- Decision Tree for quantitative risk scoring (data-driven, probability output)
- Flag disagreements for human review

> **See `reports/accuracy_comparison.md`** for the complete written analysis.